In [6]:
# Read in the dataset

import pandas as pd
import numpy as np

df = pd.read_csv("modelling_dataset.csv")
df.head()

,Zone_Int_ID,Time_Stamp,Mean_Severity,Mean_Distance(mi),Mean_Temp(F),Mean_Humidity(%),Mean_Pressure(in),Mean_Visibility(mi),Mean_Wind(mph),Mean_Precip(in),Mean_Duration(min),Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Hour,Day_of_Week,Month,Weekend,Holiday,City_Houston,City_LA,City_Miami,Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility,Day_Night,Accident_Count
0,0,2016-06-14 20:00:00,2.0,0.000,86.0,66.0,29.84,8.0,9.2,0.0,46.0,False,False,False,False,False,True,False,True,20,1,6,False,0,True,False,False,True,False,False,False,False,False,False,1,1
1,0,2016-06-21 18:00:00,2.0,0.000,82.4,79.0,30.02,10.0,9.2,0.0,30.0,False,False,False,False,False,False,False,True,18,1,6,False,0,True,False,False,True,False,False,False,False,False,False,1,1
2,0,2016-06-21 20:00:00,2.0,0.000,84.2,66.0,30.01,10.0,8.1,0.0,30.0,False,True,False,False,False,False,False,True,20,1,6,False,0,True,False,False,True,False,False,False,False,False,False,1,1
3,0,2016-06-22 12:00:00,2.0,0.033,93.2,49.0,30.03,10.0,5.8,0.0,360.0,False,False,False,True,False,False,False,False,12,2,6,False,0,True,False,False,False,True,False,False,False,False,False,1,1
4,0,2016-06-22 14:00:00,2.0,0.000,94.1,46.5,30.00,10.0,8.1,0.0,30.0,False,False,False,False,False,False,False,True,14,2,6,False,0,True,False,False,False,True,False,False,False,False,False,1,2


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351950 entries, 0 to 351949
Data columns (total 36 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Zone_Int_ID           351950 non-null  int64  
 1   Time_Stamp            351950 non-null  object 
 2   Mean_Severity         351950 non-null  float64
 3   Mean_Distance(mi)     351950 non-null  float64
 4   Mean_Temp(F)          351950 non-null  float64
 5   Mean_Humidity(%)      351950 non-null  float64
 6   Mean_Pressure(in)     351950 non-null  float64
 7   Mean_Visibility(mi)   351950 non-null  float64
 8   Mean_Wind(mph)        351950 non-null  float64
 9   Mean_Precip(in)       351950 non-null  float64
 10  Mean_Duration(min)    351950 non-null  float64
 11  Amenity               351950 non-null  bool   
 12  Crossing              351950 non-null  bool   
 13  Give_Way              351950 non-null  bool   
 14  Junction              351950 non-null  bool   
 15  

In [ ]:
y = df['Accident_Count']

mean = y.mean()
variance = y.var()

print(f"Mean:     {mean:.2f}")
print(f"Variance: {variance:.2f}")
print(f"Ratio:    {variance/mean:.2f}")

# Distribution of counts
print(f"\nCount distribution:")
print(y.value_counts().sort_index().head(10))
print(f"\nZeros: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"Max count in a single bucket: {y.max()}")

In [ ]:
d2 = df.copy()
d2['Time_Stamp'] = pd.to_datetime(d2['Time_Stamp'])

# The zone exposure is how much oppertunity there was for accidents to occur since some zones are observed more than others
zone_exposure = d2.groupby('Zone_Int_ID')['Time_Stamp'].apply(
    lambda x: x.dt.date.nunique()
).reset_index(name='Observed_Days')

# Each day has 12 two-hour slots
zone_exposure['Total_2hr_Slots'] = zone_exposure['Observed_Days'] * 12
zone_exposure['log_exposure'] = np.log(zone_exposure['Total_2hr_Slots'].clip(lower=1))

# Merge into modelling_df
df = d2.merge(zone_exposure[['Zone_Int_ID', 'log_exposure']], 
                                   on='Zone_Int_ID', how='left')

# print(df['log_exposure'].describe())
df

In [ ]:
feature_cols = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday',
    'City_Houston', 'City_LA', 'City_Miami',
    'Mean_Temp(F)', 'Mean_Humidity(%)', 'Mean_Pressure(in)',
    'Mean_Visibility(mi)', 'Mean_Wind(mph)', 'Mean_Precip(in)',
    'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway',
    'Station', 'Stop', 'Traffic_Signal',
    'Weather_Clear', 'Weather_Cloudy', 'Weather_Rain/Drizzle',
    'Weather_Stormy', 'Weather_Visibility', 'Weather_Dust/Windy',
    'Day_Night'
]

# Convert all boolean columns to int
bool_cols = df[feature_cols].select_dtypes(include='bool').columns.tolist()
print(f"Converting: {bool_cols}")
df[bool_cols] = df[bool_cols].astype(int)

# Verify
print(df[feature_cols].dtypes)

In [ ]:
df.head()

In [ ]:
print(f"modelling_df shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import TweedieRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

mean = df['Accident_Count'].mean()
variance = df['Accident_Count'].var()
alpha = (variance - mean) / mean
print(f"Using alpha: {alpha:.3f}")

feature_cols = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday',
    'City_Houston', 'City_LA', 'City_Miami',
    'Mean_Temp(F)', 'Mean_Humidity(%)', 'Mean_Pressure(in)',
    'Mean_Visibility(mi)', 'Mean_Wind(mph)', 'Mean_Precip(in)',
    'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway',
    'Station', 'Stop', 'Traffic_Signal',
    'Weather_Clear', 'Weather_Cloudy', 'Weather_Rain/Drizzle',
    'Weather_Stormy', 'Weather_Visibility', 'Weather_Dust/Windy',
    'Day_Night'
]

X = df[feature_cols].values
y = df['Accident_Count'].values
weights = df['log_exposure'].values

# Scale features for better convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_scaled, y, weights, test_size=0.2, random_state=42
)

# power=1.5 is between Poisson (1.0) and Gamma (2.0) — approximates Negative Binomial
model = TweedieRegressor(power=1.5, alpha=alpha, link='log', max_iter=1000)
model.fit(X_train, y_train, sample_weight=w_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Mean Absolute Error: {np.mean(np.abs(y_test - y_pred)):.3f}")
print(f"Mean predicted count: {y_pred.mean():.3f}")
print(f"Mean actual count:    {y_test.mean():.3f}")

# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df)